In [22]:
FOLDER = 'synthetic'

In [23]:
import pandas as pd
import json
import os
import glob
pd.set_option('display.max_columns', None)

def compile_variations_log(base_dir=FOLDER):
    log_files = glob.glob(os.path.join(base_dir, 'variation_*', 'config_log.json'))
    records = []
    
    for fpath in log_files:
        with open(fpath, 'r') as f:
            data = json.load(f)

        data['variation_id'] = os.path.basename(os.path.dirname(fpath))
        records.append(data)
        
    if not records:
        print("Nenhuma variação encontrada.")
        return pd.DataFrame()
        
    df   = pd.DataFrame(records)
    cols = ['variation_id', 'iou'] + [c for c in df.columns if c not in ['variation_id', 'iou']]
    df   = df[cols]
    
    df = df.sort_values(by='iou', ascending=False).reset_index(drop=True)
    return df


df_variations = compile_variations_log()
df_variations.head(10)

Nenhuma variação encontrada.


""


In [10]:
log_files = glob.glob(os.path.join('synthetic', 'variation_*', 'config_log.json'))
variation = 'variation_13'

for path in log_files:
    curr = path.split('/')[1].strip()

    if curr != variation:
        continue

    with open(path, 'r') as f:
        data = json.load(f)  


del data['shape'], data['margin'], data['iou']
print(json.dumps(data, indent=4))

{
    "layerRange": [
        72,
        154
    ],
    "layerThickness": [
        1,
        7
    ],
    "foldCount": [
        12,
        26
    ],
    "foldSigma": [
        7,
        74
    ],
    "foldAmplitude": [
        -28,
        34
    ],
    "foldDamping": 2.729108861461884,
    "foldBaseShift": [
        -0.846091274399015,
        2.9883234275397896
    ],
    "shearOffset": [
        -1.5974688247171969,
        4.808438156341225
    ],
    "shearGradient": [
        -0.3992332906561768,
        0.23740125496512832
    ],
    "faultCount": [
        0,
        8
    ],
    "faultThrow": [
        3,
        36
    ],
    "faultDipAngle": [
        54,
        89
    ],
    "faultRoughness": 2.323079447168227,
    "faultRoughSigma": 5.614677961692463,
    "faultDecaySigma": [
        9,
        119
    ],
    "faultZoneWidth": 0.9495823003844502,
    "faultThreshold": 0.9302376560072076,
    "faultCurveProb": 0.5680622818868736,
    "faultCurveMax": 6.09612048417798

In [20]:
import ast
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.optimize import differential_evolution
from sklearn.ensemble import RandomForestRegressor


class MultivariateIoUOptimizer:
    def __init__(self, dataframe: pd.DataFrame, target_col: str = 'iou', drop_cols: list = None):
        if drop_cols is None:
            drop_cols = ['variation_id']
        self.df = dataframe.copy()
        self.target_col = target_col
        self.drop_cols = drop_cols
        self.model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        self.features = []
        self.X = None
        self.y = None

    def _parse_list_columns(self):
        original_cols = list(self.df.columns)
        for col in original_cols:
            if col == self.target_col or col in self.drop_cols:
                continue
            
            if self.df[col].dropna().empty:
                continue
                
            first_valid = self.df[col].dropna().iloc[0]
            
            if isinstance(first_valid, str) and first_valid.strip().startswith('[') and first_valid.strip().endswith(']'):
                self.df[col] = self.df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
                first_valid = self.df[col].dropna().iloc[0]
            
            if isinstance(first_valid, list):
                self.df[f'{col}_min'] = self.df[col].apply(lambda x: np.min(x) if isinstance(x, list) and len(x) > 0 else np.nan)
                self.df[f'{col}_max'] = self.df[col].apply(lambda x: np.max(x) if isinstance(x, list) and len(x) > 0 else np.nan)
                self.df[f'{col}_mean'] = self.df[col].apply(lambda x: np.mean(x) if isinstance(x, list) and len(x) > 0 else np.nan)
                self.df.drop(columns=[col], inplace=True)

    def _remove_zero_variance(self):
        numeric_df = self.df.select_dtypes(include=[np.number])
        variances = numeric_df.var()
        cols_to_drop = variances[variances == 0].index
        self.df.drop(columns=cols_to_drop, inplace=True)

    def preprocess(self):
        self._parse_list_columns()
        self._remove_zero_variance()
        
        self.df.drop(columns=[c for c in self.drop_cols if c in self.df.columns], inplace=True, errors='ignore')
        self.df.dropna(inplace=True)
        
        self.y = self.df[self.target_col]
        self.X = self.df.drop(columns=[self.target_col])
        self.features = self.X.columns.tolist()

    def fit_model(self):
        if self.X is None or self.y is None:
            self.preprocess()
        self.model.fit(self.X, self.y)

    def analyze_impact_and_optima(self) -> pd.DataFrame:
        bounds = [(self.X[col].min(), self.X[col].max()) for col in self.features]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            def objective(x):
                return -self.model.predict(x.reshape(1, -1))[0]
            
            res = differential_evolution(objective, bounds, seed=42)
            
        global_optimum_x = res.x
        global_optimum_iou = -res.fun

        results = []
        for i, col in enumerate(self.features):
            corr, _ = spearmanr(self.X[col], self.y)
            direction = "Aumentar melhora" if corr > 0 else "Diminuir melhora"
            if abs(corr) < 0.1:
                direction = "Neutro/Não-linear"

            results.append({
                'Feature': col,
                'Spearman_Corr': round(corr, 4),
                'Impacto_Direcional': direction,
                'Valor_Min_Historico': round(bounds[i][0], 4),
                'Valor_Max_Historico': round(bounds[i][1], 4),
                'Valor_Otimo_Estimado': round(global_optimum_x[i], 4),
                'IoU_Esperado_No_Otimo': round(global_optimum_iou, 4),
                'Feature_Importance_RF': round(self.model.feature_importances_[i], 4)
            })

        return pd.DataFrame(results).sort_values(by='Feature_Importance_RF', ascending=False).reset_index(drop=True)


optimizer = MultivariateIoUOptimizer(dataframe=df_variations)
optimizer.preprocess()
optimizer.fit_model()

df_analysis = optimizer.analyze_impact_and_optima()
df_analysis

,Feature,Spearman_Corr,Impacto_Direcional,Valor_Min_Historico,Valor_Max_Historico,Valor_Otimo_Estimado,IoU_Esperado_No_Otimo,Feature_Importance_RF
0,faultCount_min,0.6324,Aumentar melhora,0.0000,4.0000,3.8540,0.7029,0.5284
1,faultDipAngle_mean,0.6657,Aumentar melhora,30.0000,72.0000,70.2314,0.7029,0.1465
2,faultZoneWidth,-0.5578,Diminuir melhora,0.5003,3.5000,0.6505,0.7029,0.1113
3,waveletDt,-0.1634,Diminuir melhora,0.0001,0.0649,0.0087,0.7029,0.0552
4,faultDipAngle_min,0.5818,Aumentar melhora,10.0000,55.0000,46.6616,0.7029,0.0303
5,foldDamping,-0.3052,Diminuir melhora,0.0018,4.9988,0.7096,0.7029,0.0109
6,faultCount_max,-0.2865,Diminuir melhora,4.0000,14.0000,6.9245,0.7029,0.0089
7,faultCount_mean,-0.0046,Neutro/Não-linear,2.0000,8.5000,7.1906,0.7029,0.0055
8,faultCurveMax,-0.0904,Neutro/Não-linear,0.0066,9.9074,3.2726,0.7029,0.0035
9,faultThreshold,-0.0884,Neutro/Não-linear,0.1006,1.4999,0.7824,0.7029,0.0034
